# Comparison analysis (template)

Copy to `research/<group>/analysis.ipynb` on the agent branch, set `GROUP` and `PROJECT`, and run.
Everything is read **only through the W&B API** (`wandb.Api()`, the `eval-episodes-*` artifacts, run
histories); nothing local. Run interactively to explore (nothing is written to W&B), then publish with

```
python scripts/research_runs.py run-analysis --project <project> --group <group> --notebook research/<group>/analysis.ipynb
```

which checks every code cell, re-executes the notebook from scratch in an empty directory, logs the
figures and tables you kept as an `analysis` run, attaches the executed notebook to it, and writes the
Analysis draft for `publish-report --analysis`.

Conventions it applies: runs of one group; `status:superseded` skipped; variants from
`config.runner_cfg.research.variant` / `config.train_cfg.research.variant` / `config.research.variant`; the scored population is the pooled
first episodes of the `eval-episodes-*` artifacts; report raw training-seed effects and matched evaluation differences where unit identities exist.
Low-N comparisons have no automatic uncertainty interval or winner test.

In [ ]:
GROUP = "REPLACE-ME"      # W&B group = research/<group>/ = dev/agent/<group>
PROJECT = "REPLACE-ME"    # W&B project, e.g. rwm-explore
ENTITY = None             # None: WANDB_USERNAME, else the API's default entity
BASELINE = None           # None: single protocol baseline; legacy: first variant by name
EVAL_RUN_IDS = None        # explicit reviewed selection if multiple attempts/protocol versions exist

# Scored-episode metrics (profile: check `survival_rate` in the task profile before adding it).
# name: (reducer over the episode rows of one group, higher is better, is a rate of 0/1 columns)
METRICS = {
    "success_rate": (lambda d: d["success"].astype(float).mean(), True, True),
    "fall_rate": (lambda d: d["fell"].astype(float).mean(), False, True),
    "lin_vel_rmse": (lambda d: float(np.sqrt((d["lin_vel_tracking_error"] ** 2).mean())), False, False),
    "ang_vel_rmse": (lambda d: float(np.sqrt((d["ang_vel_tracking_error"] ** 2).mean())), False, False),
}
RATE_COLUMN = {"success_rate": "success", "fall_rate": "fell"}  # for k/n denominators in the draft
# Per-unit estimand versus pooled estimand. Episode rows already store an episode RMSE.
# Mean episode-RMSE difference is not a pooled-RMSE difference.
UNIT_METRICS = {"success_rate": ("success", "success_rate"), "fall_rate": ("fell", "fall_rate"),
                "lin_vel_episode_rmse": ("lin_vel_tracking_error", "lin_vel_rmse"),
                "ang_vel_episode_rmse": ("ang_vel_tracking_error", "ang_vel_rmse")}
HEADLINE = ["success_rate", "fall_rate", "lin_vel_rmse"]        # columns of the draft's table
PERF_KEYS = ["Perf/total_fps", "Perf/collection_time", "Perf/learning_time"]  # training cost history


def train_variant_seed(run):
    """(variant, seed) of a training run. Older runs have no research block: override this."""
    research = (run.config.get("train_cfg") or run.config.get("runner_cfg") or {}).get("research") or run.config.get("research") or {}
    match = re.search(r"-seed(\d+)(?:-attempt\d+)?$", run.name)
    return research.get("variant"), int(research["seed"]) if "seed" in research else (
        int(match.group(1)) if match else None)

In [ ]:
import hashlib
import json
import os
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wandb
from IPython.display import Markdown, display

assert "REPLACE-ME" not in (GROUP, PROJECT), "set GROUP and PROJECT in the parameters cell"
api = wandb.Api()
entity = ENTITY or os.environ.get("WANDB_USERNAME") or api.default_entity
path = f"{entity}/{PROJECT}"
all_runs = list(api.runs(path, filters={"group": GROUP}))
runs = [r for r in all_runs if "status:superseded" not in r.tags and "status:smoke" not in r.tags]
train = [r for r in all_runs if r.job_type == "train" and "status:smoke" not in r.tags]
evals = [r for r in runs if r.job_type == "eval"]
if EVAL_RUN_IDS is not None:
    if len(EVAL_RUN_IDS) != len(set(EVAL_RUN_IDS)) or set(EVAL_RUN_IDS) - {r.id for r in evals}:
        raise ValueError("evaluation selection contains duplicate or unavailable run IDs")
    evals = [r for r in evals if r.id in EVAL_RUN_IDS]
assert train, f"no current training runs in group {GROUP}"

FIGURES, TABLES, PROBLEMS = {}, {}, []   # what the run keeps; what undermines the comparison


def keep_fig(slug, fig):
    FIGURES[slug] = fig
    return fig


def keep_table(slug, df):
    TABLES[slug] = df.reset_index(drop=True)
    return df

## 1. Provenance and protocol

Training runs, their budget and commit; every training run needs a scored evaluation, on one protocol
and one checkpoint rule. Anything that differs lands in `PROBLEMS` and in the draft's limitations.

In [ ]:
def research_metadata(run):
    return (run.config.get("train_cfg") or run.config.get("runner_cfg") or {}).get("research") or run.config.get("research") or {}


def recorded_cost(value):
    """Missing, invalid and non-finite measurements remain explicitly unavailable."""
    return float(value) if isinstance(value, (int, float, np.number)) and not isinstance(value, bool) and np.isfinite(value) and value >= 0 else float("nan")


eval_info = {}
selected_ids = set()
for r in evals:
    rs = r.config["research"]
    identity = (rs["variant"], int(rs["seed"]))
    if identity in eval_info:
        raise ValueError("multiple evaluated attempts/versions for one cell; explicitly select reviewed EVAL_RUN_IDS, first valid identical attempt and protocol")
    if rs.get("protocol_id"):
        contract = rs.get("protocol_contract") or {}
        canonical = {k: v for k, v in contract.items() if k != "protocol_id"}
        identifier = "p-" + hashlib.sha256(json.dumps(canonical, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest()[:24]
        if identifier != rs["protocol_id"]:
            raise ValueError("evaluation protocol contract content mismatch")
        linked = [t for t in train if t.id == rs.get("train_run_id")]
        if len(linked) != 1:
            raise ValueError("evaluation's exact training attempt is missing")
        tr = research_metadata(linked[0])
        for key in ("variant", "seed", "cell_id", "attempt", "hypothesis_id", "protocol_id", "protocol_version", "task"):
            if rs.get(key) is None or rs[key] != tr.get(key):
                raise ValueError("training/evaluation identity mismatch: " + key)
        commit = tr.get("commit") or (tr.get("runtime_fingerprint") or {}).get("commit")
        if not commit or rs.get("commits") != [commit]:
            raise ValueError("training/evaluation source commit mismatch")
    else:
        name = r.name.removesuffix("__eval")
        linked = [t for t in train if t.name == name or t.name.endswith("_" + name)]
        if len(linked) != 1:
            raise ValueError("legacy evaluation training attempt is missing or ambiguous")
    if r.state != "finished" or {"status:invalid", "status:failed"} & set(r.tags):
        PROBLEMS.append("selected evaluation is unfinished or invalid: " + r.name)
    if {"status:invalid", "status:failed"} & set(linked[0].tags):
        PROBLEMS.append("selected training attempt is invalid: " + linked[0].name)
    selected_ids.add(linked[0].id)
    eval_info[identity] = {"checkpoint": os.path.splitext(os.path.basename(rs.get("checkpoint") or ""))[0],
        "protocol": rs.get("protocol_id") or (rs.get("protocol") or {}).get("slug"),
        "population": (rs.get("protocol") or {}).get("population"), "run": r}

# Every attempt remains visible. Only the explicitly linked attempt contributes scientific evidence.
inv = []
for r in train:
    variant, seed = train_variant_seed(r)
    rs = research_metadata(r)
    inv.append({"run": r.name, "run_id": r.id, "variant": variant, "train_seed": seed,
                "attempt": rs.get("attempt", 1), "state": r.state, "selected": r.id in selected_ids,
                "max_iterations": (r.config.get("train_cfg") or r.config.get("runner_cfg") or {}).get("max_iterations"),
                "last_step": getattr(r, "lastHistoryStep", None),
                "wall_time_h": recorded_cost(r.summary.get("_runtime")) / 3600,
                "commit": (rs.get("commit") or (rs.get("runtime_fingerprint") or {}).get("commit") or "")[:12]})
attempts = pd.DataFrame(inv).sort_values(["variant", "train_seed", "attempt"])
keep_table("attempts", attempts)
inventory = attempts[attempts["selected"]].copy()
if inventory.empty:
    raise ValueError("no training attempts have scored evaluations")
variants = sorted(inventory["variant"].dropna().unique())
contracts = [i["run"].config["research"].get("protocol_contract") for i in eval_info.values()]
protocol_baselines = {tuple(c["baseline_variants"]) for c in contracts if c}
if len(protocol_baselines) > 1:
    raise ValueError("baseline declarations differ across protocols")
if BASELINE is None and protocol_baselines:
    baselines = next(iter(protocol_baselines))
    if len(baselines) != 1:
        raise ValueError("multiple approved baselines: set BASELINE explicitly")
    BASE = baselines[0]
else:
    BASE = BASELINE or variants[0]
if BASE not in variants or (protocol_baselines and BASE not in next(iter(protocol_baselines))):
    raise ValueError("selected baseline has no scored evidence or differs from protocol")

inventory["evaluated"] = True
inventory["checkpoint"] = [eval_info[(v, s)]["checkpoint"] for v, s in
                           zip(inventory["variant"], inventory["train_seed"])]
keep_table("inventory", inventory)
seeds = inventory.groupby("variant")["train_seed"].apply(lambda s: sorted(s))
if len({tuple(v) for v in seeds}) > 1:
    PROBLEMS.append("variants were evaluated with different training seed sets: " +
                    "; ".join(f"{k}: {v}" for k, v in seeds.items()))
for column, label in (("max_iterations", "training budgets (iterations)"), ("commit", "training commits"),
                      ("checkpoint", "evaluated checkpoints")):
    if not inventory[column].replace("", np.nan).notna().any():
        PROBLEMS.append(f"{label} are not recorded for any training run, so they cannot be compared")
    elif inventory[column].nunique(dropna=False) > 1:
        PROBLEMS.append(f"{label} differ across runs: {sorted(inventory[column].dropna().astype(str).unique())}")
if (inventory["state"] != "finished").any():
    PROBLEMS.append("selected training attempts are unfinished")
missing = sorted(set(zip(attempts["variant"], attempts["train_seed"])) - set(eval_info))
if missing:
    PROBLEMS.append("no scored evaluation for training cells: " + str(missing))
if len({i["protocol"] for i in eval_info.values()}) > 1:
    PROBLEMS.append("evaluation protocols differ: " + str(sorted({str(i["protocol"]) for i in eval_info.values()})))
if {i["population"] for i in eval_info.values()} - {"first"}:
    PROBLEMS.append("the scored population is not 'first episodes' for every evaluation")
for c in contracts:
    if c:
        declared = {(v, s) for v in c["variants"] for s in c["seeds"]}
        if set(eval_info) != declared:
            PROBLEMS.append("scored training cell matrix differs from approved protocol")
runtime_records = [research_metadata(r).get("runtime_fingerprint") or {} for r in train if r.id in selected_ids]
for field in ("python", "packages", "isaaclab_commit", "cuda_runtime", "gpu_driver", "environment_hashes", "asset_hashes"):
    values = {json.dumps(fp[field], sort_keys=True) for fp in runtime_records if fp.get(field) is not None}
    if len(values) > 1:
        PROBLEMS.append("runtime fingerprint mismatch: " + field)
# Protocol-declared custom diagnostics also participate in seed and paired tables.
from research_workflow.verification import declared_metric_reducers

BASE_UNIT_COLUMNS = {column for column, _ in UNIT_METRICS.values()}
REDUCER_SPECS = {}
for contract in contracts:
    if not contract:
        continue
    ev = contract["evaluation"]
    for metric, spec in declared_metric_reducers(ev).items():
        if metric in {"lin_vel_tracking_rmse", "ang_vel_tracking_rmse"}:
            continue
        column, reduction = spec["column"], spec["reduction"]
        if spec.get("complement"):
            column = "__" + metric + "_outcome"
        if metric in METRICS:
            if REDUCER_SPECS.get(metric, (column, reduction)) != (column, reduction):
                raise ValueError(f"metric {metric!r} is defined differently across protocol contracts in this group")
            continue
        REDUCER_SPECS[metric] = (column, reduction)
        def reducer(frame, column=column, reduction=reduction):
            values = frame[column].astype(float)
            return float(np.sqrt((values ** 2).mean())) if reduction == "rms" else float(values.mean())
        METRICS[metric] = (reducer, ev["primary_metric"]["direction"] == "higher" if metric == ev["primary_metric"]["name"] else None, reduction == "rate")
        UNIT_METRICS[metric + "_episode_value"] = (column, metric)
        if reduction == "rate":
            RATE_COLUMN[metric] = column
    primary = {"lin_vel_tracking_rmse": "lin_vel_rmse", "ang_vel_tracking_rmse": "ang_vel_rmse"}.get(ev["primary_metric"]["name"], ev["primary_metric"]["name"])
    if primary in METRICS and primary not in HEADLINE:
        HEADLINE.append(primary)
inventory


## 2. Scored episodes

The pooled first episodes of every environment, from the `eval-episodes-<variant>-seed<seed>` artifacts.
A `slice` is the level or difficulty part of the cell name (`L4-seed41` -> `L4`); this repository has no levels or difficulties, so cells are `seed41` and there is a single pooled slice.

In [ ]:
from research_workflow.metric_resolution import MetricResolver
# Explicit evaluation-attempt selection applies to scientific aggregation, while all
# attempts remain available in the inventory and spend diagnostics.
METRIC_RESOLVER = MetricResolver([r for r in all_runs
    if (r.job_type != "train" or r.id in selected_ids)
    and (r.job_type != "eval" or r.id in {i["run"].id for i in eval_info.values()})])

frames = []
EVIDENCE_ARTIFACTS = []
for (variant, seed), info in sorted(eval_info.items()):
    run = info["run"]
    expected_name = "eval-episodes-" + run.name.removesuffix("__eval")
    attached = [a for a in run.logged_artifacts() if a.type == "eval-episodes"
                and a.name.rsplit("/", 1)[-1].split(":")[0] == expected_name]
    if len(attached) != 1:
        raise ValueError("evaluation must attach exactly one scored-episode artifact: " + run.name)
    artifact = attached[0]
    if not re.search(r":v\d+$", artifact.name):
        raise ValueError("scored evidence artifact must have an immutable version")
    EVIDENCE_ARTIFACTS.append({"eval_run_id": run.id, "artifact": artifact.name, "digest": artifact.digest})
    rows = artifact.get("episodes").get_dataframe()
    if rows.empty or not (rows["episode_id"] == 0).all():
        raise ValueError("scored evidence must contain only the declared first episodes")
    contract = run.config["research"].get("protocol_contract")
    reducers_here = declared_metric_reducers(contract["evaluation"]) if contract else {}
    for metric, spec in reducers_here.items():
        if spec.get("complement"):
            rows["__" + metric + "_outcome"] = 1 - rows[spec["column"]].astype(float)
    # Required columns come from this run's own contract plus the template's fixed base
    # metrics -- never the union across every contract version seen in the group, since an
    # amended contract's extra metric must not fail an earlier contract's valid evidence.
    required_columns = BASE_UNIT_COLUMNS | {
        "__" + metric + "_outcome" if spec.get("complement") else spec["column"]
        for metric, spec in reducers_here.items()
    }
    if required_columns - set(rows.columns):
        raise ValueError("scored evidence lacks declared metric columns")
    metric_columns = sorted(required_columns)
    if not np.isfinite(rows[metric_columns].astype(float).to_numpy()).all():
        raise ValueError("non-finite scored metric rows")
    if (rows[["lin_vel_tracking_error", "ang_vel_tracking_error"]] < 0).any().any():
        raise ValueError("episode RMSE cannot be negative")
    if not rows[list(RATE_COLUMN.values())].astype(float).isin([0, 1]).all().all():
        raise ValueError("rate outcomes must be binary")
    contract = run.config["research"].get("protocol_contract")
    if contract:
        ev = contract["evaluation"]
        if ev["episode_sampling"] != "first_episode_per_env":
            raise ValueError("template only supports declared first-episode scoring")
        contexts = {f"L{level}-seed{s}": {"seed": s, "level": level, "difficulty": None}
                    for level in ev["levels"] for s in ev["seeds"]}
        contexts.update({f"{d}-seed{s}": {"seed": s, "level": None, "difficulty": d}
                         for d in ev["difficulties"] for s in ev["seeds"]})
        if not contexts:
            contexts = {f"seed{s}": {"seed": s, "level": None, "difficulty": None} for s in ev["seeds"]}
        if set(rows["cell"]) != set(contexts):
            raise ValueError("artifact evaluation cell matrix differs from protocol")
        for cell, group in rows.groupby("cell"):
            if len(group) != ev["num_envs"] or set(group["env_id"]) != set(range(ev["num_envs"])):
                raise ValueError("artifact environment population differs from protocol")
            factors = ev["paired_factors"]
            if factors:
                if "eval_unit_id" not in group:
                    raise ValueError("artifact lacks declared paired identity")
                expected_ids = []
                for row in group.to_dict("records"):
                    values = {**contexts[cell], **row}
                    if any(values.get(f) is None for f in factors):
                        raise ValueError("artifact lacks declared paired factors")
                    expected_ids.append(json.dumps({f: values[f] for f in factors}, sort_keys=True, separators=(",", ":")))
                if expected_ids != group["eval_unit_id"].tolist() or len(set(expected_ids)) != len(expected_ids):
                    raise ValueError("artifact paired identity differs from declared factors")
    frames.append(rows.assign(variant=variant, train_seed=seed))
keep_table("evidence_artifacts", pd.DataFrame(EVIDENCE_ARTIFACTS))
episodes = pd.concat(frames, ignore_index=True)
episodes["slice"] = episodes["cell"].str.replace(r"-?seed\d+$", "", regex=True).replace("", "all cells")
episodes["eval_seed"] = episodes["cell"].str.extract(r"seed(\d+)$")[0].astype(int)
SLICES = sorted(episodes["slice"].unique(), key=lambda s: (s[0] != "L", int(s[1:]) if s[0] == "L" and s[1:].isdigit() else 0, s))

per_seed = pd.DataFrame([
    {"variant": v, "train_seed": s, "slice": sl, "episodes": len(g), **{m: f(g) for m, (f, _, _) in METRICS.items()}}
    for (v, s, sl), g in episodes.groupby(["variant", "train_seed", "slice"])])
everything = pd.DataFrame([
    {"variant": v, "train_seed": s, "slice": "ALL", "episodes": len(g), **{m: f(g) for m, (f, _, _) in METRICS.items()}}
    for (v, s), g in episodes.groupby(["variant", "train_seed"])])
per_seed = pd.concat([everything, per_seed], ignore_index=True)
keep_table("per_seed", per_seed)

metric_cols = list(METRICS)
eligible_seed_rows = per_seed.loc[[all(
    METRIC_RESOLVER.comparison(row.variant, row.variant,
        per_seed.loc[(per_seed["variant"] == row.variant) & (per_seed["slice"] == row.slice), "train_seed"].tolist(), metric)["status"] == "compatible"
    for metric in METRICS) for row in per_seed.itertuples()]]
aggregate = (eligible_seed_rows.groupby(["slice", "variant"])
             .agg(training_seeds=("train_seed", "nunique"), episodes=("episodes", "sum"),
                  **{f"{m}_{stat}": (m, stat) for m in metric_cols for stat in ("mean", "min", "max")})
             .reset_index())
keep_table("aggregate", aggregate)
aggregate[aggregate["slice"] == "ALL"]

## 3. Effects and practical significance

Describe each training seed and paired evaluation units separately. Low-N evidence does not support precise intervals. Criterion classification requires the predeclared contract and constraints; never select a winner.


In [ ]:
rows = []
for sl in ["ALL", *SLICES]:
    part = per_seed[per_seed["slice"] == sl]
    base = part[part["variant"] == BASE].set_index("train_seed")
    for variant in variants:
        if variant == BASE:
            continue
        mine = part[part["variant"] == variant].set_index("train_seed")
        for metric, (_, higher, _) in METRICS.items():
            common = mine.index.intersection(base.index)
            eligibility = METRIC_RESOLVER.comparison(variant, BASE, list(common), metric)
            if eligibility["status"] != "compatible":
                PROBLEMS.append(f"incompatible or unverified comparison: {variant}/{BASE}/{metric}")
                continue
            effects = mine.loc[common, metric] - base.loc[common, metric]
            rows.append({"slice": sl, "metric": metric, "variant": variant, "baseline": BASE,
                         "delta": float(effects.mean()), "seed_effects": effects.tolist(),
                         "training_seeds": len(common), "evidence": "low-N; raw effects, no interval"})
effects = pd.DataFrame(rows, columns=["slice", "metric", "variant", "baseline", "delta", "seed_effects", "training_seeds", "evidence"])
keep_table("effects", effects)

paired_rows = []
if "eval_unit_id" in episodes and episodes["eval_unit_id"].notna().all():
    for variant in variants:
        if variant == BASE:
            continue
        keys = ["train_seed", "eval_unit_id"]
        a = episodes[episodes["variant"] == variant]
        b = episodes[episodes["variant"] == BASE]
        if a.duplicated(keys).any() or b.duplicated(keys).any():
            PROBLEMS.append("duplicate matched evaluation units; paired evidence invalid")
            continue
        matched = a.merge(b, on=keys, suffixes=("_a", "_b"), validate="one_to_one")
        if len(matched) != len(a) or len(matched) != len(b):
            PROBLEMS.append("missing matched evaluation units; pairing incomplete")
            continue
        for metric, (column, aggregate_metric) in UNIT_METRICS.items():
            for seed, part in matched.groupby("train_seed"):
                if METRIC_RESOLVER.comparison(variant, BASE, [seed], aggregate_metric)["status"] != "compatible":
                    continue
                a_values, b_values = part[column + "_a"].astype(float), part[column + "_b"].astype(float)
                delta = a_values - b_values
                aggregate_difference = (METRICS[aggregate_metric][0](part.rename(columns={column + "_a": column}))
                                        - METRICS[aggregate_metric][0](part.rename(columns={column + "_b": column})))
                paired_rows.append({"variant": variant, "baseline": BASE, "train_seed": seed,
                    "metric": metric, "estimand": "mean of variant-minus-baseline episode outcomes",
                    "aggregate_metric": aggregate_metric, "aggregate_difference": aggregate_difference,
                    "units": len(delta), "mean_difference": delta.mean(),
                    "median_difference": delta.median(), "fraction_positive": (delta > 0).mean(),
                    "fraction_negative": (delta < 0).mean(), "fraction_tied": (delta == 0).mean()})
else:
    if any((i["run"].config.get("research", {}).get("protocol_contract") or {}).get("evaluation", {}).get("paired_factors") for i in eval_info.values()):
        PROBLEMS.append("declared pairing has no verified eval_unit_id; paired evidence invalid")
    elif not any(i["run"].config.get("research", {}).get("protocol_id") for i in eval_info.values()):
        PROBLEMS.append("legacy workflow or unpaired evaluation: no verified eval_unit_id; paired effects unavailable")
paired = pd.DataFrame(paired_rows)
keep_table("paired_effects", paired)

criteria = []
for info in eval_info.values():
    contract = info["run"].config.get("research", {}).get("protocol_contract")
    if contract:
        for category in ("support_criteria", "contradiction_criteria", "inconclusive_criteria"):
            for criterion in contract[category]:
                criteria.append({"criterion": criterion, "category": category,
                    "classification": "invalid_due_to_protocol_or_provenance" if PROBLEMS else "inconclusive"})
# Replace inconclusive only after checking the declared metric, meaningful effect and every constraint.
keep_table("criteria", pd.DataFrame(criteria).drop_duplicates())
effects


In [ ]:
fig, axes = plt.subplots(1, len(HEADLINE), figsize=(4.2 * len(HEADLINE), 3.4), constrained_layout=True, squeeze=False)
offset = dict(zip(variants, np.linspace(-0.25, 0.25, len(variants)), strict=True))
for ax, metric in zip(axes[0], HEADLINE, strict=True):
    for variant in variants:
        sub = per_seed[(per_seed["variant"] == variant) & (per_seed["slice"] != "ALL")]
        xs = [SLICES.index(s) + offset[variant] for s in sub["slice"]]
        ax.scatter(xs, sub[metric], s=16, alpha=0.75, label=variant)
    ax.set_xticks(range(len(SLICES)), SLICES, fontsize=7)
    ax.set_title(f"{metric}: one point per training seed and slice", fontsize=8)
    ax.grid(alpha=0.3)
axes[0][0].legend(fontsize=7)
keep_fig("per_seed_by_slice", fig)

## 4. Cost

Training wall time (W&B runtime) and the `Perf/*` history. Extra costs outside these runs (RWM
pretraining, data collection) are not in W&B: say so in `NOTES`.

In [ ]:
cost_rows = []
for r in train:
    variant, seed = train_variant_seed(r)
    history = r.history(keys=PERF_KEYS, samples=2000, pandas=True)
    cfg = r.config.get("train_cfg") or r.config.get("runner_cfg") or {}
    declared_cost = r.summary.get("research/cost") or {}
    cost_rows.append({"run_id": r.id, "attempt": research_metadata(r).get("attempt", 1), "state": r.state, "protocol_id": research_metadata(r).get("protocol_id"), "selected": r.id in selected_ids, "variant": variant, "train_seed": seed, "environment_steps": recorded_cost(declared_cost.get("environment_steps")),
                      "updates": recorded_cost(declared_cost.get("updates", cfg.get("max_iterations") if r.state == "finished" else None)),
                      "gpu_hours": recorded_cost(declared_cost.get("gpu_hours")),
                      "parameters": recorded_cost(declared_cost.get("parameters")),
                      "inference_latency_ms": recorded_cost(declared_cost.get("inference_latency_ms")),
                      "inference_memory_mb": recorded_cost(declared_cost.get("inference_memory_mb")),
                      "teacher_pretraining_gpu_hours": recorded_cost(declared_cost.get("teacher_pretraining_gpu_hours")),
                      "wall_time_h": recorded_cost(r.summary.get("_runtime")) / 3600,
                      **{k.split("/")[-1]: recorded_cost(pd.to_numeric(history[k], errors="coerce").replace([np.inf, -np.inf], np.nan).mean()) if k in history else float("nan") for k in PERF_KEYS}})
cost_attempts = pd.DataFrame(cost_rows)
keep_table("cost_attempts", cost_attempts)
cost = cost_attempts.loc[cost_attempts["selected"] & (cost_attempts["state"] == "finished")].copy()
# Resource totals above retain failures; seed means below require a compatible cohort.
cost = cost.loc[[METRIC_RESOLVER.comparison(row.variant, row.variant,
    cost.loc[cost["variant"] == row.variant, "train_seed"].tolist(), "success_rate")["status"] == "compatible"
    for row in cost.itertuples()]]
keep_table("cost_selected_seeds", cost)
cost_fields = [c for c in cost.columns if c not in {"variant", "train_seed", "run_id", "attempt", "state", "protocol_id", "selected"}]
cost_by_variant = cost.groupby("variant")[cost_fields].agg(["mean", "std", "count"])
# Totals include every attempt, including failed attempts; unknown spend stays unknown.
spend_rows = []
for variant, attempts in cost_attempts.groupby("variant", dropna=False):
    row = {"variant": variant, "attempts": len(attempts), "failed_attempts": int((attempts["state"] != "finished").sum())}
    for key in ("wall_time_h", "gpu_hours", "environment_steps", "updates", "teacher_pretraining_gpu_hours"):
        values = attempts[key]
        row[key + "_known_subtotal"] = values.sum(min_count=1)
        row[key + "_unknown_attempts"] = int(values.isna().sum())
        row[key + "_total"] = values.sum() if values.notna().all() else float("nan")
    spend_rows.append(row)
keep_table("cost_all_attempt_spend", pd.DataFrame(spend_rows))
cost_by_variant.columns = ["_".join(c) for c in cost_by_variant.columns]
keep_table("cost", cost_by_variant.reset_index())

fig, axes = plt.subplots(1, 3, figsize=(11, 3.0), constrained_layout=True)
for ax, col in zip(axes, ("wall_time_h", "learning_time", "collection_time"), strict=True):
    mean, spread = cost.groupby("variant")[col].mean().reindex(variants), cost.groupby("variant")[col].std().reindex(variants)
    ax.bar(range(len(variants)), mean, yerr=spread.fillna(0), capsize=3)
    ax.set_xticks(range(len(variants)), variants, fontsize=7)
    ax.set_title(col + " (mean +- std over training seeds)", fontsize=8)
keep_fig("training_cost", fig)
cost_by_variant

## 5. Videos

Scored videos are the ones the evaluation recorded (`video/*` on the `eval` runs). They are listed here,
not judged: look at sampled frames before writing anything qualitative into `NOTES`.

In [ ]:
videos = pd.DataFrame([
    {"variant": v, "train_seed": s, "scored_videos": sum(k.startswith("video/") for k in info["run"].summary.keys())}
    for (v, s), info in sorted(eval_info.items())])
keep_table("videos", videos)
videos

## 6. Analysis draft

Every number below is computed above. Write only judgment in `NOTES` and never restate a number there
(the cell refuses percentages, fractions and decimals); refer to a table or figure instead.

In [ ]:
NOTES = {
    "evidence": "",     # interpretation of raw seed effects and paired differences
    "tradeoffs": "",    # what a better variant gives up (cost, other slices, other metrics)
    "confounders": "",  # limitations the data above cannot show (extra costs, one seed, ...)
    "candidates": "",   # candidate(s) worth considering, with criteria; none is a valid answer
    "next": "",         # a proposed next check, only if the evidence cannot support a decision
}

In [ ]:
NUMBER = re.compile(r"\d+(\.\d+)?\s?%|\b\d+\s?/\s?\d+\b|\b\d+\.\d+\b")
bad = {k: NUMBER.findall(v) for k, v in NOTES.items() if NUMBER.search(v)}
if bad:
    raise ValueError(f"NOTES restate numbers; point to a table or figure instead: {sorted(bad)}")


def describe(variant, sl, metric):
    """One cell of the table: pooled value with its denominator and the spread over training seeds."""
    sub = per_seed[(per_seed["variant"] == variant) & (per_seed["slice"] == sl)]
    if METRIC_RESOLVER.comparison(variant, variant, sub["train_seed"].tolist(), metric)["status"] != "compatible":
        return "No compatible aggregate; inspect per_seed diagnostics"
    rows = episodes if sl == "ALL" else episodes[episodes["slice"] == sl]
    rows = rows[rows["variant"] == variant]
    f, _, is_rate = METRICS[metric]
    spread = f"{sub[metric].min():.3g} to {sub[metric].max():.3g} over {len(sub)} seed(s)"
    if is_rate:
        k = int(rows[RATE_COLUMN[metric]].astype(bool).sum())
        return f"{k}/{len(rows)} first episodes ({k / len(rows):.1%}); {spread}"
    return f"{f(rows):.3g} ({len(rows)} first episodes); {spread}"


def table(sl):
    head = "| Variant | " + " | ".join(HEADLINE) + " | Training wall time (h) |\n|---|" + "---|" * (len(HEADLINE) + 1)
    lines = []
    for v in variants:
        wall = cost.loc[cost["variant"] == v, "wall_time_h"]
        lines.append(f"| {v} | " + " | ".join(describe(v, sl, m) for m in HEADLINE) + f" | {wall.mean():.3g} +- {wall.std(ddof=0):.2g} |")
    return head + "\n" + "\n".join(lines)


def draft():
    protocols = sorted({str(i["protocol"]) for i in eval_info.values()})
    out = [f"### Comparison\nGroup `{GROUP}`; population: first episodes per environment, pooled over cells; "
           f"protocol {', '.join(protocols)}; training seeds {', '.join(f'{k}: {v}' for k, v in seeds.items())}; "
           f"baseline `{BASE}`.\n", table("ALL")]
    for sl in SLICES:
        if sl != "all cells":
            out.append(f"\nSlice `{sl}`\n\n" + table(sl))
    out.append("\n### Effects and hypothesis criteria")
    out += [f"- `{r.variant}` vs `{r.baseline}` on {r.metric} ({r.slice}): mean difference {r.delta:+.3g}; raw training-seed effects {r.seed_effects}; {r.evidence}"
            for r in effects.itertuples()]
    out.append("Paired evaluation effects: see paired_effects table; no winner selected.")
    out += [NOTES["evidence"], "\n### Trade-offs", NOTES["tradeoffs"], "\n### Confounders / limitations"]
    out += [f"- {p}" for p in PROBLEMS] or ["- No protocol, budget, commit or seed differences found by the checks above."]
    out += [NOTES["confounders"], "\n### Candidate(s) worth considering", NOTES["candidates"] or "_None named._"]
    if NOTES["next"]:
        out += ["\n### Suggested next check", NOTES["next"]]
    return "\n".join(line for line in out if line is not None)


DRAFT = draft()
display(Markdown(DRAFT))
print("<<<ANALYSIS_DRAFT\n" + DRAFT + "\nANALYSIS_DRAFT>>>")

In [ ]:
# Deterministic evidence precedes any generated scientific interpretation.
from research_workflow.scientific_review import build_evidence

SCIENTIFIC_EVIDENCE = build_evidence(
    all_runs, project=PROJECT, group=GROUP, entity=entity,
    tables=TABLES, problems=PROBLEMS, baseline=BASE, resolver=METRIC_RESOLVER,
    hypothesis=json.loads(os.environ.get("RESEARCH_HYPOTHESIS", "null")),
)
SCIENTIFIC_EVIDENCE["analysis_code"] = json.loads(os.environ.get("RESEARCH_ANALYSIS_CODE", "null"))
from research_workflow.scientific_review import identifier
SCIENTIFIC_EVIDENCE["evidence_id"] = identifier("evidence", SCIENTIFIC_EVIDENCE)
print("<<<SCIENTIFIC_EVIDENCE\n" + json.dumps(SCIENTIFIC_EVIDENCE, allow_nan=False) + "\nSCIENTIFIC_EVIDENCE>>>")


## 7. Keep

Runs only under `run-analysis` (which sets `RESEARCH_ANALYSIS_LOG=1`); interactive runs never write to
W&B. The run holds the figures and tables kept above; the CLI then attaches this executed notebook.

In [ ]:
if os.environ.get("RESEARCH_ANALYSIS_LOG") == "1":
    source = json.loads(os.environ.get("RESEARCH_SOURCE", "{}"))
    run = wandb.init(project=PROJECT, entity=entity, group=GROUP, job_type="analysis", name=f"{GROUP}__analysis",
                     save_code=True, tags=["dirty"] if source.get("dirty") else [],
                     config={"research": {"comparison": GROUP, "notebook": f"research/{GROUP}/analysis.ipynb",
                                          "source": source, "baseline": BASE, "train_runs": [r.name for r in train],
                                          "eval_runs": [r.name for r in evals], "evidence_artifacts": EVIDENCE_ARTIFACTS}})
    kept = {f"analysis/fig/{slug}": wandb.Image(fig) for slug, fig in FIGURES.items()}
    kept |= {f"analysis/table/{slug}": wandb.Table(dataframe=df) for slug, df in TABLES.items()}
    run.log(kept)
    run.summary["analysis/scientific_evidence"] = SCIENTIFIC_EVIDENCE
    run.summary["analysis/draft_md"] = DRAFT
    run.summary["analysis/problems"] = len(PROBLEMS)
    run.finish()
    print("logged", sorted(kept))
else:
    print("interactive run: nothing written to W&B; publish with scripts/research_runs.py run-analysis")